# Meeting Minutes from an Audio File

An actual end-to-end product: turning a raw audio recording of a city council
meeting into structured meeting minutes (summary, discussion points, takeaways,
and action items with owners).

**The audio:** an extract from Denver City Council meeting minutes. Original
source dataset: https://huggingface.co/datasets/huuuyeah/meetingbank (audio
files at https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).
I'm using a shorter extract for this exercise, but recording something of my own
would work just as well.

**The pipeline, in two steps:**
1. **Transcribe** the audio to text -- trying both an open-source model
   (Whisper via Hugging Face) and OpenAI's hosted transcription API, to compare.
2. **Analyze & summarize** the transcript into proper meeting minutes using an
   open-source LLM (Llama 3.1), run locally in 4-bit quantization.

Runs on a Colab T4 GPU.


## Reminder to self: the misleading CUDA error

If I see an error like:

> `Runtime error: CUDA is required but not available for bitsandbytes...`

that's misleading -- not really a package version issue. Usually means Colab
swapped out my runtime underneath me. Fix:

1. `Runtime` menu -> Disconnect and delete runtime
2. Reload the notebook fresh, `Edit` menu -> Clear All Outputs
3. Reconnect to a new T4 (top-right button)
4. Check "View resources" to confirm the GPU is actually attached
5. Re-run all cells from the top, starting with the pip installs


## Setup


In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers


In [ ]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch


In [ ]:
# Constants

LLAMA = "meta-llama/Llama-3.1-8B-Instruct"


## Getting the audio file onto the Colab

Connecting Colab to Google Drive, so the audio file can just live there instead
of re-uploading it every session. Before running this: download the extract
(`denver_extract.mp3`) from
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing,
then place it in a `llms` folder on my Google Drive.


In [ ]:
drive.mount("/content/drive")
audio_filename = "/content/denver_extract.mp3"


In [ ]:
# Sign in to Hugging Face Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the audio file

audio_file = open(audio_filename, "rb")


## Step 1: Transcribe the audio

Trying two different approaches to transcription, to compare them directly.

### Option 1: open-source transcription with Whisper

Using Hugging Face's `pipeline` with OpenAI's Whisper model (open-sourced, run
locally here rather than through OpenAI's API). `return_timestamps=True` lets
Whisper handle audio longer than its usual 30-second chunk limit by
transcribing in timestamped segments internally.


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)


In [ ]:
open_source_transcription = transcription


### Option 2: OpenAI's hosted transcription API

Same audio, this time sent to OpenAI's `gpt-4o-mini-transcribe` model via the
API instead of running anything locally.


In [ ]:
# Sign in to OpenAI using Secrets in Colab

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)
transcription = openai_client.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)


Comparing both transcriptions side by side -- worth checking how closely they
agree, and whether one handles names, numbers, or pauses noticeably better than
the other.


In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))


## Step 2: Analyze & generate the report

Now turning the raw transcript into structured minutes. The system prompt sets
the output format (markdown, no code blocks, specific sections), and the user
prompt embeds the actual transcript text plus the exact structure I want:
summary with attendees/location/date, discussion points, takeaways, and action
items with owners.


In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


Same 4-bit quantization setup as before, needed to fit Llama 3.1 8B comfortably
on a T4.


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)


Loading the tokenizer and model, building the input from the chat template, and
generating -- streaming the output so I can watch the minutes get written in
real time rather than waiting for the whole 2000-token generation to finish.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)


Decoding the raw output tokens back into readable markdown text.


In [ ]:
response = tokenizer.decode(outputs[0])


And finally, rendering it properly as formatted markdown instead of a raw string
-- this is the actual finished meeting minutes.


In [ ]:
display(Markdown(response))


## Ideas to try next

- Swap in a different open model for the summarization step (Phi, Qwen, etc.
  from the models notebook) and compare the quality of the generated minutes.
- Stream the output into a proper Gradio UI instead of printing to the
  notebook -- `TextIteratorStreamer` combined with a background thread is the
  way to do this without blocking the UI while generation runs.
- Try this on a recording of my own instead of the Denver extract, to see how
  well it generalizes to different audio quality, accents, and meeting formats.
